# The ROC curve and AUC

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("data/diabetes.csv")
print(df.shape, df["Outcome"].value_counts().to_dict())
X, y = df.drop(columns="Outcome"), df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Probabilities, then a threshold

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
p = lr.predict_proba(X_test)[:, 1]          # probability of diabetes
for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    tn, fp, fn, tp = confusion_matrix(y_test, (p >= t).astype(int)).ravel()
    print(f"t = {t}: TP {tp:2d} FN {fn:2d} FP {fp:2d} TN {tn:3d}   TPR {tp / (tp + fn):.3f}  FPR {fp / (fp + tn):.3f}")

## The ROC curve and the threshold closest to (0, 1)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, p)
best = np.argmin(np.hypot(fpr, 1 - tpr))
print("thresholds tried:", len(thresholds))
print(f"best threshold {thresholds[best]:.3f}: TPR {tpr[best]:.3f}, FPR {fpr[best]:.3f}")

## AUC: comparing two models

In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print("logistic regression AUC", round(roc_auc_score(y_test, p), 3))
print("decision tree AUC      ", round(roc_auc_score(y_test, dt.predict_proba(X_test)[:, 1]), 3))